# Tugas Pertemuan 13 - Deep Learning & NLP Dasar

**Nama:** Nabil Fakhrezy  
**NIM:** 240401010286  
**Program Studi:** Informatika

## Materi
Pengantar topik lanjutan Data Science, yaitu **Deep Learning** dan **Natural Language Processing (NLP)**.

Pada praktikum ini, saya mengerjakan dua bagian utama:
1. Klasifikasi data non-linear menggunakan Neural Network sederhana.
2. Analisis sentimen ulasan produk menggunakan TF-IDF dan Logistic Regression.


## 1. Import Library

Library yang digunakan meliputi `numpy`, `pandas`, `matplotlib`, `seaborn`, `scikit-learn`, dan `tensorflow/keras`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

sns.set_theme(style="whitegrid")
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)


## 2. Generate dan Eksplorasi Dataset Non-Linear

Dataset `make_moons` berisi dua kelas berbentuk bulan sabit. Pola seperti ini sulit dipisahkan dengan satu garis lurus.


In [ ]:
X, y = make_moons(n_samples=300, noise=0.2, random_state=42)

df_moons = pd.DataFrame({
    "x1": X[:, 0],
    "x2": X[:, 1],
    "label": y
})

print("Shape dataset:", df_moons.shape)
display(df_moons.head())

print("Distribusi label:")
display(df_moons["label"].value_counts().rename_axis("label").reset_index(name="jumlah"))

plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_moons, x="x1", y="x2", hue="label", palette="coolwarm", alpha=0.8)
plt.title("Dataset Non-Linear make_moons")
plt.xlabel("Fitur X1")
plt.ylabel("Fitur X2")
plt.show()


**Interpretasi:**

Dua kelas pada dataset terlihat saling melengkung dan bertautan. Karena itu, model linear sederhana tidak cukup ideal. Neural Network dengan activation function non-linear dapat membentuk batas keputusan yang lebih fleksibel.


## 3. Train-Test Split


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_tr.shape)
print("X_test :", X_te.shape)
print("y_train:", y_tr.shape)
print("y_test :", y_te.shape)


Data dibagi menjadi 80% training dan 20% testing. Parameter `stratify=y` menjaga proporsi kelas tetap seimbang.


## 4. Membangun Neural Network Sederhana

Arsitektur model:
- Dense 16 neuron dengan ReLU
- Dense 8 neuron dengan ReLU
- Dense 1 neuron dengan Sigmoid


In [ ]:
model = Sequential([
    Dense(16, activation="relu", input_shape=(2,)),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


ReLU digunakan pada hidden layer agar model dapat mempelajari pola non-linear. Sigmoid digunakan pada output karena kasusnya klasifikasi biner.


## 5. Melatih Model Neural Network


In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_tr,
    y_tr,
    epochs=100,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

print("Jumlah epoch yang dijalankan:", len(history.history["loss"]))


Proses training dilakukan selama beberapa epoch. Early stopping membantu menghentikan training ketika performa validasi tidak lagi membaik.


## 6. Evaluasi Model Neural Network


In [ ]:
loss, acc = model.evaluate(X_te, y_te, verbose=0)
y_prob = model.predict(X_te, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print(f"Loss data uji    : {loss:.4f}")
print(f"Akurasi data uji : {acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_te, y_pred))

cm = confusion_matrix(y_te, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix Neural Network")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.show()


**Interpretasi:**

Akurasi menunjukkan seberapa banyak data testing yang berhasil diklasifikasikan dengan benar. Confusion matrix memperlihatkan jumlah prediksi benar dan salah pada masing-masing kelas.


## 7. Visualisasi Kurva Pembelajaran


In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_df["accuracy"], label="Training")
plt.plot(history_df["val_accuracy"], label="Validasi")
plt.title("Kurva Akurasi")
plt.xlabel("Epoch")
plt.ylabel("Akurasi")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_df["loss"], label="Training")
plt.plot(history_df["val_loss"], label="Validasi")
plt.title("Kurva Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()


Jika akurasi training dan validasi saling mendekat, model cenderung belajar dengan baik. Jika training jauh lebih tinggi dibanding validasi, model berpotensi overfitting.


## 8. Visualisasi Decision Boundary Neural Network


In [ ]:
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

grid = np.c_[xx.ravel(), yy.ravel()]
pred_grid = model.predict(grid, verbose=0).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, pred_grid, levels=30, cmap="coolwarm", alpha=0.35)
sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=y, palette="coolwarm", edgecolor="black")
plt.title("Decision Boundary Neural Network pada Dataset make_moons")
plt.xlabel("Fitur X1")
plt.ylabel("Fitur X2")
plt.show()


Decision boundary yang melengkung menunjukkan bahwa neural network mampu menyesuaikan diri dengan pola data non-linear.


## 9. Menyiapkan Dataset Ulasan Produk

Dataset ulasan terdiri dari 40 ulasan sintetis berbahasa Indonesia, dengan label `1` untuk positif dan `0` untuk negatif.


In [ ]:
ulasan = [
    "Barangnya bagus banget, pengiriman cepat",
    "Kualitas jelek, tidak sesuai deskripsi",
    "Sangat puas, akan beli lagi",
    "Kecewa, barang rusak saat sampai",
    "Recommended, harga sesuai kualitas",
    "Buruk sekali, tidak sesuai ekspektasi",
    "Produk original dan berfungsi dengan baik",
    "Pengiriman lama dan kemasan penyok",
    "Pelayanan ramah dan respon cepat",
    "Barang cacat, saya sangat kecewa",
    "Kualitas mantap dan harga terjangkau",
    "Tidak awet, baru dipakai sudah rusak",
    "Sesuai pesanan dan sangat memuaskan",
    "Warna berbeda dari foto produk",
    "Seller amanah, barang cepat sampai",
    "Produk tidak berfungsi sama sekali",
    "Packing rapi dan barang bagus",
    "Saya menyesal membeli produk ini",
    "Harga murah tapi kualitas tetap bagus",
    "Barang palsu dan mengecewakan",
    "Pengalaman belanja sangat menyenangkan",
    "Respon penjual lambat dan tidak membantu",
    "Produk nyaman digunakan sehari-hari",
    "Kualitas bahan buruk dan mudah rusak",
    "Sangat direkomendasikan untuk dibeli",
    "Barang tidak sesuai ukuran yang dipesan",
    "Pengiriman cepat, produk sesuai harapan",
    "Sangat buruk, tidak akan beli lagi",
    "Desain bagus dan performa memuaskan",
    "Paket datang terlambat dan rusak",
    "Barang berfungsi normal dan memuaskan",
    "Kualitas mengecewakan untuk harga segini",
    "Produk bagus, cocok untuk kebutuhan saya",
    "Tidak sesuai deskripsi, sangat kecewa",
    "Penjual cepat merespon dan membantu",
    "Barang kurang bagus dan tidak tahan lama",
    "Suka sekali dengan kualitas produknya",
    "Produk gagal digunakan sejak pertama dibuka",
    "Mantap, kualitas sesuai dengan harga",
    "Pelayanan buruk dan barang tidak layak"
]

label = [
    1, 0, 1, 0, 1, 0, 1, 0, 1, 0,
    1, 0, 1, 0, 1, 0, 1, 0, 1, 0,
    1, 0, 1, 0, 1, 0, 1, 0, 1, 0,
    1, 0, 1, 0, 1, 0, 1, 0, 1, 0
]

df_ulasan = pd.DataFrame({
    "ulasan": ulasan,
    "label": label
})

df_ulasan["sentimen"] = df_ulasan["label"].map({1: "Positif", 0: "Negatif"})

print("Shape dataset ulasan:", df_ulasan.shape)
display(df_ulasan.head(10))

print("Distribusi sentimen:")
display(df_ulasan["sentimen"].value_counts().rename_axis("sentimen").reset_index(name="jumlah"))


Dataset teks ini digunakan untuk melatih model analisis sentimen sederhana. Data dibuat seimbang antara ulasan positif dan negatif.


## 10. Text Vectorization dengan TF-IDF

TF-IDF digunakan untuk mengubah teks menjadi angka agar dapat diproses oleh model machine learning.


In [ ]:
tfidf = TfidfVectorizer(
    lowercase=True,
    max_features=100,
    ngram_range=(1, 2)
)

X_text = tfidf.fit_transform(df_ulasan["ulasan"])
y_text = df_ulasan["label"]

print("Shape matriks TF-IDF:", X_text.shape)
print("Jumlah fitur kata:", len(tfidf.get_feature_names_out()))
print("Contoh fitur kata:")
print(tfidf.get_feature_names_out()[:20])


TF-IDF memberi bobot pada kata yang khas dalam ulasan. Kata seperti `bagus`, `kecewa`, `rusak`, dan `memuaskan` dapat membantu model mengenali sentimen.


## 11. Melihat Kata dengan Skor TF-IDF Tertinggi


In [ ]:
tfidf_df = pd.DataFrame(
    X_text.toarray(),
    columns=tfidf.get_feature_names_out()
)

top_terms = tfidf_df.mean().sort_values(ascending=False).head(15)

display(top_terms.rename("rata_rata_tfidf").reset_index().rename(columns={"index": "kata"}))

plt.figure(figsize=(9, 5))
sns.barplot(x=top_terms.values, y=top_terms.index)
plt.title("Top 15 Kata berdasarkan Rata-rata TF-IDF")
plt.xlabel("Rata-rata TF-IDF")
plt.ylabel("Kata")
plt.show()


Kata dengan nilai TF-IDF tinggi dapat dianggap sebagai kata penting dalam kumpulan ulasan.


## 12. Train-Test Split untuk Data Teks


In [ ]:
Xt_tr, Xt_te, yt_tr, yt_te = train_test_split(
    X_text,
    y_text,
    test_size=0.2,
    random_state=42,
    stratify=y_text
)

print("X_train text:", Xt_tr.shape)
print("X_test text :", Xt_te.shape)
print("y_train text:", yt_tr.shape)
print("y_test text :", yt_te.shape)


Data teks dibagi menjadi training dan testing dengan stratifikasi agar komposisi positif dan negatif tetap seimbang.


## 13. Latih Model Klasifikasi Sentimen


In [ ]:
model_sentimen = LogisticRegression(max_iter=1000)
model_sentimen.fit(Xt_tr, yt_tr)

pred_sentimen = model_sentimen.predict(Xt_te)
akurasi_sentimen = accuracy_score(yt_te, pred_sentimen)

print(f"Akurasi model sentimen: {akurasi_sentimen:.3f}")
print("\nClassification Report:")
print(classification_report(yt_te, pred_sentimen, target_names=["Negatif", "Positif"]))

cm_sentimen = confusion_matrix(yt_te, pred_sentimen)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm_sentimen,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=["Negatif", "Positif"],
    yticklabels=["Negatif", "Positif"]
)
plt.title("Confusion Matrix Analisis Sentimen")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.show()


Logistic Regression dipakai sebagai baseline karena sederhana dan cocok untuk teks yang sudah direpresentasikan dengan TF-IDF.


## 14. Uji Prediksi Kalimat Baru


In [ ]:
kalimat_baru = [
    "Pelayanan sangat memuaskan dan ramah",
    "Barang rusak dan kualitasnya sangat buruk",
    "Produk bagus, sesuai harapan saya",
    "Saya kecewa karena barang tidak sesuai"
]

pred_baru = model_sentimen.predict(tfidf.transform(kalimat_baru))
prob_baru = model_sentimen.predict_proba(tfidf.transform(kalimat_baru))

hasil_prediksi = pd.DataFrame({
    "kalimat": kalimat_baru,
    "prediksi_label": pred_baru,
    "sentimen": ["Positif" if p == 1 else "Negatif" for p in pred_baru],
    "prob_negatif": prob_baru[:, 0],
    "prob_positif": prob_baru[:, 1]
})

display(hasil_prediksi.round(3))


Model dapat memprediksi sentimen kalimat baru berdasarkan kata-kata yang pernah dipelajari dari data training.


## 15. Kesimpulan

Pada praktikum Pertemuan 13 ini, saya mempelajari dua topik lanjutan dalam Data Science, yaitu Deep Learning dan Natural Language Processing.

Pada bagian Deep Learning, Neural Network sederhana berhasil digunakan untuk mengklasifikasikan dataset non-linear `make_moons`. Dataset ini tidak mudah dipisahkan dengan garis lurus, sehingga activation function non-linear seperti ReLU membantu model membentuk batas keputusan yang lebih fleksibel.

Pada bagian NLP, teks ulasan produk berhasil diubah menjadi representasi numerik menggunakan TF-IDF, kemudian diklasifikasikan menggunakan Logistic Regression. Model dapat membedakan ulasan positif dan negatif secara sederhana.

Secara umum, Deep Learning berguna untuk mempelajari pola kompleks, sedangkan NLP memungkinkan data teks dianalisis menggunakan pendekatan machine learning. Namun, hasil praktikum ini masih terbatas karena dataset yang digunakan bersifat sintetis dan ukurannya kecil.
